# Customer Churn Prediction with a Simple Feed‑Forward Neural Network (FFNN)

This notebook demonstrates a simple feed‑forward neural network to solve a common business problem: predicting customer churn.

Why this matters for business:
- Retaining customers is often cheaper than acquiring new ones.
- A churn probability helps prioritise retention campaigns (discounts, outreach).

Featured:
- Create a small, realistic synthetic churn dataset with business features.
- Preprocess numeric and categorical features.
- Train a Keras Sequential FFNN to predict churn probability.
- Evaluate with Accuracy, Precision, Recall, F1, ROC‑AUC; view ROC curve and confusion matrix.
- Score a hypothetical customer and interpret the result.


In [ ]:
# Optional: install dependencies if running in a fresh environment
%pip install -q tensorflow scikit-learn pandas numpy matplotlib seaborn


## 1) Imports and configuration


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

np.random.seed(42)
tf.random.set_seed(42)
sns.set(style="whitegrid")


## 2) Create a small, realistic synthetic churn dataset

Here I simulate a subscription business with features such as tenure, monthly charges, contract type, support tier, number of support tickets, add‑ons, and region.

Churn logic (probabilistic): higher churn for short tenure, high monthly charges relative to tenure, month‑to‑month contracts, many support tickets, and no add‑ons.


In [ ]:
N = 4000
rng = np.random.default_rng(42)

tenure_months = rng.integers(1, 61, size=N)
monthly_charges = rng.normal(loc=60, scale=20, size=N).clip(5, 200)
total_charges = monthly_charges * tenure_months + rng.normal(0, 50, size=N)
contract_type = rng.choice(['Month-to-month', 'One year', 'Two year'], size=N, p=[0.6, 0.25, 0.15])
support_tier = rng.choice(['Standard', 'Priority', 'Premium'], size=N, p=[0.6, 0.3, 0.1])
num_tickets = rng.poisson(lam=2.0, size=N)
has_addons = rng.choice(['Yes', 'No'], size=N, p=[0.4, 0.6])
region = rng.choice(['NA', 'EMEA', 'APAC', 'LATAM'], size=N, p=[0.4, 0.25, 0.25, 0.10])

# Base churn propensity
z = (
    1.5
    - 0.04 * tenure_months
    + 0.008 * (monthly_charges - 60)
    + 0.25 * (num_tickets)
    - 0.3 * (has_addons == 'Yes').astype(float)
)

z += np.where(contract_type == 'Month-to-month', 0.6, 0.0)
z += np.where(contract_type == 'One year', -0.2, 0.0)
z += np.where(contract_type == 'Two year', -0.5, 0.0)

z += np.where(support_tier == 'Priority', -0.1, 0.0)
z += np.where(support_tier == 'Premium', -0.25, 0.0)

# Region small effects
z += np.where(region == 'LATAM', 0.05, 0.0)
z += np.where(region == 'EMEA', -0.03, 0.0)

# Convert logit to probability
prob = 1 / (1 + np.exp(-z))
churn = rng.binomial(1, prob)

data = pd.DataFrame({
    'tenure_months': tenure_months,
    'monthly_charges': monthly_charges,
    'total_charges': total_charges,
    'contract_type': contract_type,
    'support_tier': support_tier,
    'num_tickets': num_tickets,
    'has_addons': has_addons,
    'region': region,
    'churn': churn
})
data.head()


## 3) Train/validation split and preprocessing

- One‑hot encode categorical variables.
- Standardize numeric variables.
- Keep a transformer so we can preprocess new customers consistently.


In [ ]:
target = 'churn'
X = data.drop(columns=[target])
y = data[target].astype(int).values

categorical_cols = ['contract_type', 'support_tier', 'has_addons', 'region']
numeric_cols = [c for c in X.columns if c not in categorical_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse=False), categorical_cols),
    ]
)

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)
X_train_proc.shape, X_test_proc.shape


## 4) Build a simple FFNN in Keras

Architecture:
- Input layer (dimension = preprocessed feature count)
- Hidden Dense(32) ReLU + Dropout(0.2)
- Hidden Dense(16) ReLU
- Output Dense(1) Sigmoid (churn probability)

I use Binary Cross‑Entropy loss and the Adam optimizer. EarlyStopping monitors validation loss.


In [ ]:
input_dim = X_train_proc.shape[1]
model = keras.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()


## 5) Train the model


In [ ]:
early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
history = model.fit(
    X_train_proc, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=64,
    callbacks=[early_stop],
    verbose=0
)
import pandas as pd
pd.DataFrame(history.history).head()


## 6) Evaluate on the test set


In [ ]:
# Predict probabilities and labels
y_prob = model.predict(X_test_proc, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print(f'Accuracy:  {acc:.3f}')
print(f'Precision: {prec:.3f}')
print(f'Recall:    {rec:.3f}')
print(f'F1-score:  {f1:.3f}')
print(f'ROC-AUC:   {auc:.3f}')

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Churn','Churn'])
disp.plot(values_format='d', cmap='Blues')
plt.title('Confusion Matrix (Threshold = 0.5)')
plt.show()

# ROC curve
fpr, tpr, thr = roc_curve(y_test, y_prob)
plt.figure(figsize=(6,4))
plt.plot(fpr, tpr, label=f'AUC = {auc:.3f}')
plt.plot([0,1],[0,1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()


## 7) Score a hypothetical customer and interpret

I create a single example, pass it through the same preprocessor, predict churn probability, and explain how a business might act on it.


In [ ]:
new_customer = pd.DataFrame([{
    'tenure_months': 3,
    'monthly_charges': 85,
    'total_charges': 3 * 85,
    'contract_type': 'Month-to-month',
    'support_tier': 'Standard',
    'num_tickets': 4,
    'has_addons': 'No',
    'region': 'NA'
}])

new_proc = preprocessor.transform(new_customer)
prob = float(model.predict(new_proc, verbose=0).ravel()[0])
print(f'Predicted churn probability: {prob:.2%}')

if prob >= 0.5:
    action = (
        'High risk: Consider targeted retention — proactive outreach, limited‑time discount, or service review.'
    )
else:
    action = (
        'Lower risk: Maintain standard engagement; offer add‑ons or loyalty perks to increase stickiness.'
    )
print('Suggested action:', action)
new_customer


## Notes
- This is a small synthetic dataset; real‑world performance depends on data quality and volume.
- Try adjusting the network size, learning rate, and class weights if your dataset is imbalanced.
- For deployment, persist both `preprocessor` and `model` and apply them consistently to incoming data.
